# WSJ 2027 - Gruppindelning Rundresa

Assign confirmed rundresa deltagare into groups of exactly 36.

## Rules
1. **Exactly 36 per group** (remainder group allowed)
2. **Geographic proximity** — participants should live close to each other (Hilbert curve sort)
3. **Friend wish** — at least ONE of friend_1/friend_2 in same group (soft goal)
4. **Max 8 from same kår** per group (hard constraint)
5. **Diversity** — age (14-17) and sex should be as evenly spread as possible

In [1]:
import sys
sys.path.insert(0, '/config/notebooks/wsj27')
import wsj27_utils as u

# Fetch and parse all participants
raw_data = u.fetch_participants()
df, skipped = u.build_participant_dataframe(raw_data)

Fetched 1972 participants
Confirmed: 1901, Unconfirmed: 33, Cancelled: 38
Total confirmed participants: 1900
Skipped: 33 unconfirmed, 1 wrong age/no DOB

By category:
category
deltagare    1585
ist           293
cmt            22

By travel type:
travel
rundresa      1257
direktresa     414
egen_resa      207
other           22

Friend wishes:
  With friend 1 member no: 1109
  With friend 2 member no: 656
  With friend 1 name (text): 86
  With friend 2 name (text): 60

Skipped participants:
  DELTAGARE wrong age: Ann-Ida Bergsten born 1983-07-25 (age 44)


In [2]:
GROUP_SIZE = 36

# Filter to rundresa deltagare only
df_rundresa = df[df['travel'] == 'rundresa'].copy().reset_index(drop=True)
print(f"=== Rundresa deltagare ===")
print(f"Participants: {len(df_rundresa)}")

# Assign coordinates and Hilbert sort
u.assign_coordinates(df_rundresa)
df_sorted = u.add_hilbert_index(df_rundresa)

n_full = len(df_sorted) // GROUP_SIZE
remainder = len(df_sorted) % GROUP_SIZE
total_groups = n_full + (1 if remainder > 0 else 0)
print(f"\nGroups: {n_full} x {GROUP_SIZE} + 1 x {remainder} = {total_groups} total")
print(f"\nBy region:")
print(df_sorted['region'].value_counts().to_string())
print(f"\nBy age:")
print(df_sorted['age'].value_counts().sort_index().to_string())
print(f"\nBy sex:")
print(df_sorted['sex'].map(u.SEX_MAP).value_counts().to_string())

=== Rundresa deltagare ===
Participants: 1257
With coordinates: 1185
Without coordinates (Sweden centroid): 72

Groups: 34 x 36 + 1 x 33 = 35 total

By region:
region
Region Stockholm    424
Region Södra        274
Region Västra       217
Region Norr/Mitt    182
Region Östra         86
                      2

By age:
age
14    272
15    358
16    288
17    253
18     33
19     22
20      4
21     12
22      5
23      3
24      3
25      4

By sex:
sex
Kvinna    642
Man       604
Annat       8
Okänt       3


In [3]:
# Resolve text-only friend wishes via fuzzy name matching
u.resolve_friend_wishes(df_sorted, df)
friend_wishes = u.build_friend_graph(df_sorted)

=== Text Friend Name Matching ===
Text-only wishes: 80
Matched & applied: 42
Generic wishes (not a person): 4
Unresolved (friend not in project): 34

Matched:
  Adelia Vallin -> Leo Norstedt (S:ta Maria Scoutkår) [rundresa] via fuzzy(0.96)
  Albin Åkesson -> Malte Ekström (Lomma scoutkår) [rundresa] via fuzzy(0.84)
  Alex Liljerot Priftis -> Vera Tollwé (Årsta Scoutkår) [rundresa] via fuzzy(0.86)
  Alma Stössel Weinryb -> Lotten Hellman (Södermalms Scoutkår) [rundresa] via exact
  Alma Stössel Weinryb -> Alfons Ekelin Sintorn (Södermalms Scoutkår) [rundresa] via fuzzy(0.76)+kar
  Charlie Lindberg -> Axel Lindroth (Saltsjö-Boo Scoutkår) [rundresa] via fuzzy(0.96)
  Dag Wikström -> Fjodor Yermshuk (Roslags Näsby Scoutkår) [rundresa] via fuzzy(0.97)+kar
  Dag Wikström -> Medvin Koblet (Roslags Näsby Scoutkår) [rundresa] via exact
  Ebba Stoor -> Carl-Johan Samils (Järlinden Scoutkår) [rundresa] via exact
  Elsa Ekdahl -> Sebastian Kardin (Equmenia Scout) [rundresa] via exact
  Elsie Stenf

In [4]:
# Run the full group assignment algorithm
df_sorted = u.assign_groups(df_sorted, GROUP_SIZE, friend_wishes)
total_groups = df_sorted['group'].nunique()

Participants: 1257
Groups: 34 x 36 + 1 x 33 = 35 total

=== Phase 1: Geographic sort + cut ===
  Friend satisfaction: 622/781
  Kar violations: 165
  Avg geo spread: 0.6570

=== Phase 2: Fix friend wishes ===
  Swaps: 91
  Friend satisfaction: 770/781
  Kar violations: 148
  Avg geo spread: 1.2519

=== Phase 3: Fix kar violations (geo-aware) ===
  Swaps: 140
  Kar violations: 0
  Friend satisfaction: 622/781
  Avg geo spread: 1.3603

=== Phase 2b: Re-fix friends after kar fix ===
  Swaps: 96
  Friend satisfaction: 766/781
  Kar violations: 0
  Avg geo spread: 1.8096

=== Phase 4: Diversity optimization (geo-penalized) ===
  Swaps: 281
  Diversity score: 113.03 -> 113.78
  Avg geo spread: 1.8096 -> 1.7649

=== FINAL RESULTS ===
Groups: 34 x 36 + 1 x 33
Friend satisfaction: 766/781 (98%)
Kar violations: 0
Total swaps: 608
Diversity: 113.78
Avg geo spread: 1.7649


In [5]:
# Per-group quality metrics
group_of = df_sorted['group'].values
u.print_group_metrics(df_sorted, group_of, total_groups)

Grupp Storlek  Van% MaxKar     M/K/A  14  15  16  17  AvstandKm Karer
--------------------------------------------------------------------------------
    1      36 27/27      7   14/22/0   9  11   9   7        26    14
    2      36 25/25      8   14/22/0  10  13   8   5        78    14
    3      36 26/27      8   18/18/0   7   9   6  10        10     9
    4      36 26/26      7   17/19/0  11  11   6   6        29    13
    5      36 19/19      7   14/22/0  13  12   3   5        19    15
    6      36 17/17      5   16/19/0   9   9   7   8        56    18
    7      36 18/20      6   20/16/0  11   9   7   5        38    18
    8      36 21/23      7   14/22/0  11  10   5   8       139    17
    9      36 28/29      8   18/18/0  10   7  10   8       115    14
   10      36 19/20      8   13/22/1   5  11  10   8        33    14
   11      36 19/19      7   19/17/0   7  13   8   6        46    15
   12      36 29/29      8   12/24/0   5   9  10   9        53    11
   13      36 17/18  

In [6]:
# Export CSV + JSON
OUTPUT_DIR = '/config/notebooks/wsj27'
csv_path, json_path = u.export_results(df_sorted, group_of, total_groups, OUTPUT_DIR, prefix='wsj27_rundresa')

Saved 1257 participants to /config/notebooks/wsj27/wsj27_rundresa_grupper.csv
Saved group summary to /config/notebooks/wsj27/wsj27_rundresa_grupper.json

CSV preview (first 10 rows):
 group              name member_no  age    sex                    kar                        district           region friend_1 friend_2       lat       lng
     1   Mirja Magnusson   3367087   15 Kvinna      Bunkeflo Scoutkår      Södra Skånes Scoutdistrikt     Region Södra                   55.557482 12.963234
     1         Anouk Asp   3351398   15 Kvinna         Dalby Scoutkår      Södra Skånes Scoutdistrikt     Region Södra  3355911  3386859 55.664664 13.346159
     1   Saga Lee Zander   3386859   14 Kvinna         Dalby Scoutkår      Södra Skånes Scoutdistrikt     Region Södra  3351398  3355911 55.664664 13.346159
     1    Stella Jönsson   3355911   14 Kvinna         Dalby Scoutkår      Södra Skånes Scoutdistrikt     Region Södra  3351398  3386859 55.664664 13.346159
     1   Winston Persson   33308

In [7]:
# Generate interactive map
map_path = f'{OUTPUT_DIR}/wsj_rundresa_karta.html'
u.generate_group_map_html(df_sorted, total_groups, map_path, title='WSJ 2027 Rundresa',
                          friend_wishes=friend_wishes, show_group_arcs=True)
print(f"\nAll outputs:")
print(f"  CSV:  {csv_path}")
print(f"  JSON: {json_path}")
print(f"  Map:  {map_path}")

Friend arcs: 761 (659 satisfied, 102 unsatisfied)
Group arcs: 21948 connections across 35 groups
Saved group map: /config/notebooks/wsj27/wsj_rundresa_karta.html

All outputs:
  CSV:  /config/notebooks/wsj27/wsj27_rundresa_grupper.csv
  JSON: /config/notebooks/wsj27/wsj27_rundresa_grupper.json
  Map:  /config/notebooks/wsj27/wsj_rundresa_karta.html
